# Imports and data:

In [1]:
import pandas as pd
import numpy as np
import regex as re

In [2]:
w1 = pd.read_excel("data/WINNERS 2000-2010 - 05.xlsx")
w2 = pd.read_excel("data/WINNERS 2011-2020 - 05.xlsx")
w3 = pd.read_excel("data/WINNERS 2021-2025 - 05.xlsx")
w4 = pd.read_excel("data/WINNERS 2026-2030 - 05.xlsx")

In [3]:
winners = pd.concat([w1, w2, w3, w4], ignore_index=True)

winners.shape

(43457, 10)

In [4]:
winners.dtypes

Name                   object
School                 object
Year                    int64
City                   object
Award                  object
Division               object
Category               object
School / Company       object
Division / Position    object
Dance Category         object
dtype: object

In [5]:
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [6]:
# for s in winners.School.value_counts().index:
#     print(s)

In [7]:
# for a in winners["Award"].value_counts().index:
#     print(a)

In [8]:
winners["Division"].value_counts()

Division
Senior Age Division             2837
Junior Age Division             2604
Pre-Competitive Age Division    1102
Top 12 - Senior Age Division       1
Name: count, dtype: int64

In [9]:
winners["Category"].value_counts()

Category
Classical Dance Category                   2909
Contemporary Dance Category                2874
Ensembles                                   451
Special Awards                              446
Large Ensembles                             133
Pas De Deux                                  73
Small Ensembles                              67
Classical & Contemporary Dance Category      24
Duets & Trios                                 3
Name: count, dtype: int64

In [10]:
# for i in winners["Division / Position"].value_counts().index:
#     print(i)

In [11]:
winners["Division / Position"] = winners["Division / Position"].str.replace(r"\(.*\)", "", regex=True).str.strip()
winners["Division / Position"].value_counts()

Division / Position
Junior Age Division             11795
Senior Age Division             11604
Pre-Competitive Age Division     7446
Large Ensembles                   368
Small Ensembles                   358
Pas De Deux                       293
Ensembles                         130
Special Awards                    120
Duet/Trio                          16
Name: count, dtype: int64

In [12]:
winners[winners.Name.str.contains(",", na=False)][["Name", "Category", "Division / Position"]]

,Name,Category,Division / Position
6,"Kory Glatman, NJ",NaN,NaN
87,"Melissa Lorry, MA",Contemporary Dance Category,NaN
124,"Arizona Ballet School, AZ",Special Awards,NaN
186,"April Rae, MD",NaN,NaN
207,"Bills, Bills, Bills",Large Ensembles,NaN
...,...,...,...
41363,"Strong, Calm, Slow",NaN,Duet/Trio
41910,"Diana & Acteon , Pas De Deux",NaN,Small Ensembles
41912,"Diana & Acteon , Pas De Deux",NaN,Small Ensembles
43165,"Spin, Measure, Cut",NaN,Small Ensembles


In [13]:
winners.loc[
    winners["Dance Category"].str.contains("Ensembles And Pas De Deux", na=False), 
    "Dance Category" 
] = "Ensembles & Pas De Deux"

winners.loc[
    winners["Dance Category"].str.contains(r"\bDuet\b|\bTrio\b", na=False), 
    "Dance Category"
] = "Duet/Trio"

In [14]:
winners["Dance Category"].value_counts()

Dance Category
Classical Dance Category                   17361
Contemporary Dance Category                11612
Ensembles                                   1793
Special Awards                               889
Pas De Deux                                  477
Large Ensembles                              198
Small Ensembles                              110
Classical & Contemporary Dance Category      104
Ensembles & Pas De Deux                       47
Duets & Pas De Deux                           28
Contemporary/Open Dance Category              17
Top 12                                        15
Duet/Trio                                     11
Traditional Dance Category                     9
Top 6                                          6
Name: count, dtype: int64

In [15]:
winners.isna().sum()

Name                     554
School                 36132
Year                       0
City                       0
Award                    174
Division               36913
Category               36477
School / Company        8286
Division / Position    11327
Dance Category         10780
dtype: int64

- Some awards are for the school overall rather than a person
- Some awards are for the name of the dance rather than people
- School awards wouldn't have a Dance Category or Position, which would explain some entries being null
- NaNs in certain columns can actually tell you what kind of award is being given:
    - NaN in School/Company ; School Award
- Some people don't belong in a School/Company

# Data Cleanup:

- Whenever Name and School have NaNs, there are no rows where School/Company is empty
    - So: replace School with whatever is in the School/Company for that row

In [16]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
7717,NaN,NaN,2011,"Torrington, CT",Outstanding School Award,NaN,NaN,"Washington School of Ballet, DC, USA",NaN,Special Awards
7806,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,The Art of Classical Ballet,NaN,Special Awards
7807,NaN,NaN,2011,"Tampa, FL",Outstanding School Award,NaN,NaN,America's Ballet School,NaN,Special Awards
7905,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Los Angeles Ballet Academy, CA",NaN,Special Awards
7906,NaN,NaN,2011,"San Francisco, CA",Outstanding School Award,NaN,NaN,"Westlake School for the Performing Arts, CA",NaN,Special Awards
...,...,...,...,...,...,...,...,...,...,...
42644,NaN,NaN,2026,"Boca Raton, FL",Outstanding School Award,NaN,NaN,"Stars Dance Studio, FL",Special Awards,NaN
42928,NaN,NaN,2026,"Indianapolis, IN",Outstanding School Award,NaN,NaN,"Indiana Ballet Conservatory, IN",Special Awards,NaN
43200,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"N&D Ballet, MA",Special Awards,NaN
43201,NaN,NaN,2026,"Boston-Worcester, MA",Outstanding School Award,NaN,NaN,"Koltun Ballet Boston, MA",Special Awards,NaN


In [17]:
winners[
    (winners.Name.isna()) & 
    (winners.School.isna()) &
    (winners["School / Company"].isna())
]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category


In [18]:
mask = (winners["Name"].isna()) & (winners["School"].isna())

winners.loc[mask, "School"] = winners.loc[mask, "School / Company"]
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
43452,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43453,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
43454,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
43455,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [19]:
mask =  (winners.School.isna()) & (~winners["School / Company"].isna())
winners.loc[
    mask,
    "School"    
] = winners.loc[mask, "School / Company"]

- Removing hanging whitespace from all string columns since it will effect analysis
- Turn everything uppercase, since there are inconsistent capitalization (or lack thereof)

In [20]:
winners["Name"] = winners.Name.str.strip()

In [21]:
str_cols = winners.select_dtypes(include=["object", "string"]).columns

for col in str_cols:
    mask = winners[col].notna()
    winners.loc[mask, col] = (
        winners.loc[mask, col]
        .astype(str)
        .str.replace(r"\s+,", ",", regex=True)
        .str.replace(r",\s+,", ",", regex=True)
        .str.replace(r"\(\s+(\d+)", r"(\1", regex=True)
        .str.replace(r"(\d+)\s+\)", r"\1)", regex=True)
    )

In [22]:
winners = winners.apply(lambda col: col.str.upper() if col.dtype == "object" else col)

In [23]:
names = sorted(winners.Name.dropna().unique())

with open("data/yagp_names.txt", "w") as file:
    file.writelines(
        name + "\n" for name in names
    )

- Misspellings/errors:
    - ALANAGRIFFITH = ALANA GRIFFITH
    - ALTHEA JOHNSON - STAUB
    - ANDREAJONUSAS = ANDREA JONUSAS
    - ELISA D[CARPIO = ELISA D'CARPIO
    - GABRIELLEGIRARD = GABRIELLE GIRARD
    - LOGANHILLMAN = LOGAN HILLMAN
    - MAKAI LEWIS - REES
    - MARIARODRIGUEZ = MARIA RODRIGUEZ
    - PHILIP MARTIN‐NIELSON
    - SHERY MORAY = SHERRY MORAY
    - SHI HANY HUANG = SHI HANG HUANG
    - VICTORIAHASK = VICTORIA HASK
    - WING YAN [IBBY] CHOW
- Names containing commas that need to be fixed:
    - 'CAROLIE, LAUVETZ'
    - 'ALESSANDRO FROLA, 12, MARTINA MANTOVANI, 13'
    - 'VALENTINA LEALI, MASSIMO SONZOGNI'
    - 'NORA CLEMENTE, 16, BRYCE LEE, 16'
    - 'WATMORA CASEY, TATYANA MAZUR'
    - 'JESSICA ODASZ, ANDREA ASTUTO'
    - 'KATIA GARZA, TODD ERIC ALLEN'
    - 'NADIA, TENCER'
- Different Names for the same person:
    - CHEN LI-EN, LIESL = LI-EN, LIESL CHEN ; LIESL CHEN
    - GOH JIA YI, ANU = GOH JIA YI
    - KENNETH SHELBY, JR = KENNETH DUANE SHELBY, JR ; KENNETH DUANE SHELBY JR
    - NADIA VESELOVA TENCER = NADIA VESELOVA-TENCER = NADIA, TENCER ; NADIA VESELOVA-TENCER
    - PHYLICIA, ZI YING CHEW = ZI YING CHEW
    - SOFIA, ZI XUAN LEE = ZI XUAN LEE

In [24]:
winners.loc[
    winners.Name.str.contains("ABIGAIL, BOIVIN", na=False),
    "Name"
] = "ABIGAIL BOIVIN"

winners.loc[
    winners.Name.str.contains("ALANAGRIFFITH", na=False),
    "Name"
] = "ALANA GRIFFITH"

winners.loc[
    winners.Name.str.contains("ANDREAJONUSAS", na=False),
    "Name"
] = "ANDREA JONUSAS"

winners.loc[
    winners.Name.str.contains("BRITTANY, GEORGHEGAN", na=False),
    "Name"
] = "BRITTANY GEORGHEGAN"

winners.loc[
    winners.Name.str.contains("CAROLIE, LAUVETZ", na=False),
    "Name"
] = "CAROLIE LAUVETZ"

winners.loc[
    winners.Name.str.contains("ELISA D[CARPIO", regex=False, na=False),
    "Name"
] = "ELISA D'CARPIO"

winners.loc[
    winners.Name.str.contains("GABRIELLEGIRARD", na=False),
    "Name"
] = "GABRIELLE GIRARD"

winners.loc[
    winners.Name.str.contains("GOH JIA YI, ANU", na=False),
    "Name"
] = "GOH JIA YI"

winners.loc[
    (winners.Name.str.contains("KENNETH SHELBY, JR", na=False)) |
    (winners.Name.str.contains("KENNETH DUANE SHELBY, JR", na=False)),
    "Name",
] = "KENNETH DUANE SHELBY JR"

winners.loc[
    (winners.Name.str.contains("CHEN LI-EN, LIESL", na=False)) |
    (winners.Name.str.contains("LI-EN, LIESL CHEN", na=False)),
    "Name",
] = "LIESL CHEN"

winners.loc[
    winners.Name.str.contains("LOGANHILLMAN", na=False),
    "Name"
] = "LOGAN HILLMAN"

winners.loc[
    winners.Name.str.contains("MARIARODRIGUEZ", na=False),
    "Name"
] = "MARIA RODRIGUEZ"

winners.loc[
    (winners.Name.str.contains("NADIA VESELOVA TENCER", na=False)) |
    (winners.Name.str.contains("NADIA VESELOVA-TENCER", na=False)) |
    (winners.Name.str.contains("NADIA, TENCER", na=False)),
    "Name",
] = "NADIA VESELOVA-TENCER"

winners.loc[
    winners.Name.str.contains("TAYLOR, HOGAN", na=False),
    "Name"
] = "TAYLOR HOGAN"

winners.loc[
    winners.Name.str.contains("TORI, JENNINGS", na=False),
    "Name"
] = "TORI JENNINGS"

winners.loc[
    winners.Name.str.contains("VICTORIAHASK", na=False),
    "Name"
] = "VICTORIA HASK"

winners.loc[
    winners.Name.str.contains("SOFIA, ZI XUAN LEE", na=False),
    "Name"
] = "ZI XUAN LEE"

winners.loc[
    winners.Name.str.contains("PHYLICIA, ZI YING CHEW", na=False),
    "Name"
] = "ZI YING CHEW"

winners.loc[
    winners.Name.str.contains("SHERY MORAY", na=False),
    "Name"
] = "SHERRY MORAY"

winners.loc[
    winners.Name.str.contains("SHI HANY HUANG", na=False),
    "Name"
] = "SHI HANG HUANG"

# winners.loc[
#     (winners.Name.str.contains("ABIGAIL BOIVIN", na=False)) |
#     (winners.Name.str.contains("ALANA GRIFFITH", na=False)) |
#     (winners.Name.str.contains("ALTHEA JOHNSON - STAUB", na=False)) |
#     (winners.Name.str.contains("ANDREA JONUSAS", na=False)) |
#     (winners.Name.str.contains("BRITTANY GEORGHEGAN", na=False)) |
#     (winners.Name.str.contains("ELISA D'CARPIO", na=False)) |
#     (winners.Name.str.contains("GABRIELLE GIRARD", na=False)) |
#     (winners.Name.str.contains("GOH JIA YI", na=False)) |
#     (winners.Name.str.contains("KENNETH DUANE SHELBY JR", na=False)) |
#     (winners.Name.str.contains("LIESL CHEN", na=False)) |
#     (winners.Name.str.contains("LOGAN HILLMAN", na=False)) |
#     (winners.Name.str.contains("MARIA RODRIGUEZ", na=False)) |
#     (winners.Name.str.contains("MAKAI LEWIS - REES", na=False)) |
#     (winners.Name.str.contains("PHILIP MARTIN‐NIELSON", na=False)) |
#     (winners.Name.str.contains("TAYLOR HOGAN", na=False)) |
#     (winners.Name.str.contains("TORI JENNINGS", na=False)) |
#     (winners.Name.str.contains("VICTORIA HASK", na=False)) |
#     (winners.Name.str.contains("ZI XUAN LEE", na=False)) |
#     (winners.Name.str.contains("ZI YING CHEW", na=False)) |
#     (winners.Name.str.contains("WING YAN [IBBY] CHOW", regex=False, na=False)),
#     "is_name"
# ] = "person"

In [25]:
winners[winners.Name.str.contains("ALEKSEY", na=False)]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
3333,ALEKSEY BABAYEV,ACADEMY OF INTERNATIONAL BALLET AND PERFORMING...,2006,"GREENVILLE, SC",TOP 12,PRE-COMPETITIVE AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN
18314,ALEKSEY AND INESSA PLEKHANOV,NaN,2016,"PHILADELPHIA, PA",OUTSTANDING TEACHER AWARD,NaN,NaN,NaN,NaN,SPECIAL AWARDS


- Inconsistent patterns for recording names:
    - First Name Last Name(s)
    - Last Name, First Name
    - First Name Last Name, [US state]
    - First Name Middle Name(s) and Last Name(s)
    - [1st person] & [2nd person]
    - [1st person], [2nd person]
    - [1st person] (age), [2nd person] (age)
    - (age)
    - [1st person] age, [2nd person] age (where age isn't in parentheses)
    - First Name Last Name (age), (dance that the person/people performed)
    - dance that the person/people performed, ([1st person], [2nd person])

In [26]:
# winners[winners.Name.str.contains("&", na=False)]

In [27]:
# winners[winners.Name.str.contains("& &", na=False)]

In [28]:
# winners[winners.Name.str.contains(r"\bAND\b", na=False)]

- & and "AND" separates names, but some rows have one too many & with whitespace in the middle (& &) 
    - some dances have "AND" in them too :(
        - So replace "&", "& &", and " AND" with a comma (space between name/dance to deal with whitespace)
        - Don't .str.split() on commas just yet, since dances also have commas (need to distinguish what has dancer names and what are names of dances)
- Removing ages and other dances in parentheses, but there's sometimes a hanging comma (ends in comma)

In [29]:
def split_names(string):
    if pd.isna(string):
        return string  # keep NaN
    
    if isinstance(string, str):
        return [s.strip() for s in string.split("&")]
    
    return string

winners.Name = winners.Name.str.replace(r"& &", "&")
winners.Name = winners.Name.apply(split_names)
winners = winners.explode("Name", ignore_index=True)

In [30]:
winners["Names_resolved"] = winners.Name.str.replace(r" \bAND\b", ",").str.strip()
winners["Names_resolved"] = winners.Names_resolved.str.replace(r"\(.*\)", "", regex=True).str.strip()
winners["Names_resolved"] = winners.Names_resolved.str.replace(r",$", "", regex=True).str.strip()

In [31]:
winners["Names_resolved"] = winners["Names_resolved"].str.split(",")

winners[~winners.Names_resolved.isna()]

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,Names_resolved
2,8+3,INWOOD DANCE COMPANY,2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN,[8+3]
3,CABARET,"DANCE CONSERVATORY, DE",2000,"WASHINGTON, DC",3RD PLACE (TIE),NaN,ENSEMBLES,NaN,NaN,NaN,[CABARET]
4,MAKE THEM HEAR YOU,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",2ND PLACE,NaN,ENSEMBLES,NaN,NaN,NaN,[MAKE THEM HEAR YOU]
5,CAPRICE,"DANCE CONSERVATORY, DE",2000,"WASHINGTON, DC",1ST PLACE,NaN,ENSEMBLES,NaN,NaN,NaN,[CAPRICE]
6,"KORY GLATMAN, NJ",NaN,2000,"WASHINGTON, DC",3RD PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,"[KORY GLATMAN, NJ]"
...,...,...,...,...,...,...,...,...,...,...,...
43515,GERDA'S RIVER JOURNEY,"BAYER BALLET ACADEMY, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"BAYER BALLET ACADEMY, CA",LARGE ENSEMBLES,NaN,[GERDA'S RIVER JOURNEY]
43516,DANCE FOR SIX,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN,[DANCE FOR SIX]
43517,COPPELIA FRIENDS DANCE,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"WESTLAKE SCHOOL FOR THE PERFORMING ARTS, CA",LARGE ENSEMBLES,NaN,[COPPELIA FRIENDS DANCE]
43518,KIRA FARGAS,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN,[KIRA FARGAS]


In [32]:
names = winners[winners.Name.str.contains(",", na=False) == False].Name.unique()

winners.loc[
    ((winners.Names_resolved.str[1].str.strip() + " " + winners.Names_resolved.str[0].str.strip()).isin(names)), 
    ["Names_resolved"]
] = winners.Names_resolved.str[1].str.strip() + " " + winners.Names_resolved.str[0].str.strip()

winners.loc[
    ((winners.Names_resolved.str[0].str.strip() + " " + winners.Names_resolved.str[1].str.strip()).isin(names)), 
    ["Names_resolved"]
] = winners.Names_resolved.str[0].str.strip() + " " + winners.Names_resolved.str[1].str.strip()

In [33]:
yt = winners.Names_resolved.explode()
yt[~yt.isna()]

6                 KORY GLATMAN
6                           NJ
87               MELISSA LORRY
87                          MA
124      ARIZONA BALLET SCHOOL
                 ...          
43229                  MEASURE
43229                      CUT
43244                     SPIN
43244                  MEASURE
43244                      CUT
Name: Names_resolved, Length: 553, dtype: object

- US state abbreviations followed after names need to be removed
- ages not in parentheses need to be removed

In [34]:
patterns = re.compile(
    r"^(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY|DC|\d{1,2})$"
)

# def remove_state_age(x):
#     if isinstance(x, list) and len(x) == 2:
#         first, second = x
#         if isinstance(second, str) and patterns.fullmatch(second.strip()):
#             return first.strip()
        
#     if isinstance(x, list) and len(x) == 4:
#         first, second, third, fourth = x
#         if second.strip().isdigit() & fourth.strip().isdigit():
#             return [first.strip(), third.strip()]
#     return x

def remove_state_age(x):
    if not isinstance(x, list):
        return x
    
    cleaned = []
    for i in x:
        i = i.strip()
        if patterns.fullmatch(i):
            continue
        cleaned.append(i)

    return cleaned

winners["Names_resolved"] = winners["Names_resolved"].apply(remove_state_age)

In [35]:
for i in winners.Names_resolved[~winners.Names_resolved.isna()].value_counts().index:
    if not isinstance(i, list):
        print(i)

COPPELIA PAS DE DEUX
DON QUIXOTE PAS DE DEUX
ALEXANDRIA MARX
HARLEQUINADE PAS DE DEUX
RACHEL ROHRICH
MASUMI YOSHIMOTO
PAS DE DEUX GRAND PAS CLASSIQUE
SWAN LAKE PAS DE TROIS
ODALISQUES LE CORSAIRE
RYLAND ACREE III
LAUREN LEB
JASMINE JIMISON
SILVI LYBBERT
ORLANDO BALLET SCHOOL FL
JESLYN CHEN
GENEVIEVE COOVREY
ALL AMERICAN CLASSICAL BALLET SCHOOL FL


- Classifying what entries have people's names:

In [36]:
dances_and_schools = list(winners.School.dropna().unique())
print("GINGER SMITH" in dances_and_schools)

True


In [37]:
# winners[winners.Name.str.contains(",", na=False)].Name.unique()

In [38]:
name_dances_and_schools = [
'ARIZONA BALLET SCHOOL, AZ',
'BILLS, BILLS, BILLS',
'BROTHER, CAN YOU SPARE A DIME',
'ADAMS SCHOOL OF DANCE, VT',
'PAS DE DEUX, "GISELLE", ACT III',
'ODALISQUES, LE CORSAIRE',
'GRANDE PAS DE DEUX, LE CORSAIRE',
'ENGLISH STRING SUITE, 4TH',
'ODALISQUE, LE CORSAIRE',
'CONCERTO GROSSO, 4TH MOVEMENT',
'SWAN LAKE, PAS DE TROIS',
'LA FILLE MAL GARDEE, THE CLOGS DANCE',
'MIRTH, FIRST MOVEMENT',
'WHEAT PAS DE DEUX, COPPELIA',
'SODA BALLET ACADEMY, JAPAN',
'GRAND PAS CLASSIQUE, PAS DE DEUX',
'PERHAPS, PERHAPS, PERHAPS',
'COPPELIA, THE DOLL',
'IMPACT DANCE OF ATLANTA, GA',
'MORNING STAR DANCE ACADEMY OF ATLANTA, GA',
'INTERNATIONAL BALLET ACADEMY, NC',
'AKHMEDOVA BALLET ACADEMY, MD',
'INTERNATIONAL CITY SCHOOL OF BALLET, GA',
'ORLANDO BALLET SCHOOL, FL',
'JESU, JOY!',
'PAS DE DEUX FROM FLOWER FESTIVAL AT GENZANO, CA',
"BOYS' DANCE FROM THE LIFE OF KING ARTHUR, A MEDEIVAL FANTAISIE",
'FLAMES OF PARIS, BASQUES DANCE',
'YES, JESUS LOVES EVEN ME!',
'WE, ALWAYS WE',
'GENTLY FALLING, FLOATING DOWN',
'DON QUIXOTE, PAS DE DEUX',
'COPPELIA, PAS DE DEUX',
'LE CORSAIRE, ACT III PAS DE DEUX',
'GISELLE, PEASANT PAS DE DEUX ACT I',
'ACTEON, PAS DE DEUX',
'DANCE WITH SWORDS, FROM GAYANEE',
'CUBA LINDA, CUBA HERMOSA...',
'HARLEQUINADE, PAS DE DEUX',
'FLOWER FESTIVAL IN GENZANO, PAS DE DEUX',
'POUR TOI, MARCEAU',
'SOMETHING, ANYTHING, NOTHING',
'LE CORSAIRE, ACT III',
"K-BALLET ART, LADANSE BALLET, BLUET BALLET ACADEMY, EWHA BALLET ETOILE, YONGTONG E-WHA BALLET ACADEMY, KIM HYE YOUNG BALLET ACADEMY, LEE WONJU BALLET ACADEMY, JS BALLET, POSE BALLET ACADEMY, S.M. BALLET",
"ACADEMY OF INT'L BALLET, PA",
'INTERNATIONAL BALLET THEATER ACADEMY, PA',
'TATIANA AKINFIEVA DANCE ACADEMY, DE',
'SHOKO KITAGUCHI BALLET STUDIO, JAPAN',
'LE CORSAIRE, ACT III PDD',
'SLAVE, LE CORSAIRE',
'THE DALLAS CONSERVATORY, TX',
'DANCE INSTITUTE, TX',
'ELITE CLASSICAL COACHING, TX',
'BALLET CONSERVATORY, TX',
'ALLEGRO WEST ACADEMY OF DANCE, TX',
'ART BALLET ACADEMY, TX',
'BALLET ACADEMY OF TEXAS, TX',
'BALLET CENTER OF FORT WORTH, TX',
'DALLAS BALLET CENTER, TX',
'DANCE INDUSTRY PERFORMING ARTS CENTER, TX',
'MARINA ALMAYEVA SCHOOL OF CLASSICAL BALLET, TX',
'PREMIERE BALLET ACADEMY, TX',
'KOLTUN BALLET BOSTON, MA',
'ROCKWELL DANCE CENTER, CT',
'CHANGING NOTHING, NOTHING CHANGES',
'LOOKING BACK, GOING FORWARDS',
'SUMMIT DANCE SHOPPE, MN',
'CHRISTINE RICH STUDIO DANCE ACADEMY, IL',
'ILLINOIS DANCE CONSERVATORY, IL',
'CHICAGO BALLET CONSERVATORY, IL',
'INDIANA BALLET CONSERVATORY, IL',
'NORTHPOINTE DANCE ACADEMY, OH',
'CORNELIUS, THE SELLER OF BALLOONS',
'A STORM INSIDE US, SILENCE OUTSIDE',
'CLASSICAL BALLET ACADEMY, OR',
'CLASSICAL DANCE ACADEMY, CA',
'SIMPLY BALLET, CANADA',
'EMERALD BALLET ACADEMY, WA',
'INSPIRE DANCE CENTRE, OR',
'THE DANCE CONSERVATORY WEST VANCOUVER, CANADA',
'SULTANOV RUSSIAN BALLET ACADEMY, OR',
'HOWARD W. BLAKE HIGH SCHOOL, FL',
'ALL AMERICAN CLASSICAL BALLET SCHOOL, FL',
'RENNER BALLET ACADEMY, NC',
'THE THINNING, THE LIGHT THAT BINDS US, TREMORS',
'MIRROR, MIRROR',
'CLASSICAL BALLET CENTRE, SC',
'CARY BALLET CONSERVATORY, NC',
'ACADEMY OF RUSSIAN BALLET, VA',
'UNCSA PREPARATORY DANCE PROGRAM, NC',
'METROPOLITAN SCHOOL OF THE ARTS ACADEMY, VA',
'BALLET INSTITUTE OF ATLANTA, GA',
'SWIM,SWAM,SWUM',
'DEGAS DANCE STUDIO, CA',
"BOBBIE'S SCHOOL OF PERFORMING ARTS, CA",
'NORTH COUNTY ACADEMY OF DANCE, CA',
'SOUTH COAST CONSERVATORY, CA',
'CALIFORNIA DANCE ACADEMY, CA',
'TUSTIN DANCE CENTER, CA',
'THE PREMIER BALLET SCHOOL OF ORANGE COUNTY, CA',
'SHIRLEY WINTERS BALLET, CA',
'INLAND PACIFIC BALLET ACADEMY, CA',
'THE SARASOTA CUBAN BALLET SCHOOL, FL',
"JACQUELINE'S SCHOOL OF BALLET, UT",
'BALLET WEST ACADEMY, UT',
'LOS ANGELES BALLET ACADEMY, CA',
'MOGA CONSERVATORY OF DANCE, UT',
'ROCKY MOUNTAIN BALLET ACADEMY, CO',
'LITTLETON BALLET ACADEMY, CO',
'STUDIO R BALLET, AZ',
'COLORADO BALLET ACADEMY, CO',
'CLASSICAL BALLET ACADEMY, CO',
'INDIANA BALLET CONSERVATORY, IN',
'MAIN STREET BALLET COMPANY, IL',
'DAVID MATTHEW STUDIOS, MI',
'DIANA EVANS SCHOOL OF DANCE, KY',
"VALENTINA'S SCHOOL OF BALLET, MI",
'DANCE ACADEMY OF LIBERTYVILLE, IL',
'SOUTHOLD DANCE THEATER, IN',
'TURNING POINTE DANCE STUDIO, MA',
'GREENWICH BALLET ACADEMY, CT',
'DANCING ARTS CENTER, MA',
'THE HARTT SCHOOL, CT',
'SKYRA STUDIOS, FL',
'CLASSICAL BALLET OF WNY, NY',
'VOILÀ, NOUS VOICI!',
'WITH YOU, ALWAYS',
'THE CHERRY COUNTESS, CIPOLINO',
'PAST, PRESENT AND FUTURE',
'NON, RIEN DE RIEN',
'STILL, WE FALL',
'SAME BEAT, TWICE',
'STRONG, CALM, SLOW',
'SPIN, MEASURE, CUT',

# extracted dances from mixed entries
'THE SLEEPING BEAUTY GRAND PAS ACT III',
'LA BAYADERE',
'SATANELLA',
'LA FILLE MAL GARDEE',
'HARLEQUINADE',
'COPPELIA',
'MOSZKOVSKY WALTZ',
'THE CAVALRY HALT',
'FLAMES OF PARIS',
"LE CORSAIRE, PAS D'ESCLAVE"
]


entry = name_dances_and_schools[43]
del name_dances_and_schools[43]
split_schools = [x.strip() for x in entry.split(",")]

dances_and_schools = np.unique(dances_and_schools + name_dances_and_schools + split_schools)
dances_and_schools

array(['1 DANCE ACADEMY', '1-2-3-4', '8 COUNT DANCE, OH', ...,
       '\xa0ART OF TECHNIQUE, VA', 'ÉCOLE SUPÉRIEURE DE BALLET DU QUÉBEC',
       'ÉCOLE SUPÉRIEURE DE BALLET DU QUÉBEC, CANADA'], dtype='<U170')

- Cleaning up the names column to just get people's names:

In [39]:
# winners = winners.explode("Names_resolved")

In [40]:
# winners

In [41]:
# for name in sorted(winners.Name.dropna().unique()):
#     print(name)

In [42]:
# for name in winners.Name.dropna().unique():
#     print(name)

In [43]:
# for name in sorted(
#     winners[
#         (winners.Name.str.contains(",", na=False)) & 
#         (winners.is_name == "person") 
#     ]["Name"].unique()
# ):
#     print(name)

In [44]:
# for name in sorted(winners[
#     (winners.is_name == "person") |
#     (winners.is_name == "people")
# ]["Name"].unique()):
#     print(name)

# Dancers:
- classifications: person, people

In [45]:
dancers = winners.dropna(subset="Name")
dancers.shape

(42966, 11)

## Finding a column to filter people/person from:

- Category:
    - ENSEMBLES ; DANCES
    - LARGE ENSEMBLES ; DANCES
    - PAS DE DEUX ; DANCES AND NAMES
    - SMALL ENSEMBLES ; DANCES
    - DUETS & TRIOS ; DANCES

In [46]:
sorted(dancers[dancers["Category"].str.contains(r"PAS DE DEUX", na=False)].Name.unique())

['A LITTLE LIFE',
 'ANDANTE',
 'APPASSIONATA',
 'ARABIAN PAS DE DEUX FROM THE NUTCRACKER',
 'ASHLEY LARACEY',
 'BENJAMIN HARBOR',
 'BLACK SWAN PAS DE DEUX FROM SWAN LAKE',
 'BO BUSBY',
 'CINDERELLA',
 'CLAIRE DE LUNE',
 'CONVERSATIONS WITH MYSELF',
 'DOLLS VAR. FROM "THE NUTCRACKER"',
 'DON QUIXOTE',
 'END OF MAY',
 "IT'S A GOOD DAY",
 'KYRA HOMERES',
 'LA BAYADERE',
 'LA FILLE MAL GARDEE',
 'LA FILLE MAL GARDEE PAS DE DEUX',
 'LES SYLPHIDES',
 'LIBERTANGO',
 'OFF',
 'PAS DE DEUX FROM "PAQUITA"',
 'PAS DE DEUX FROM COPPELIA',
 'PAS DE DEUX FROM DON QUIXOTE',
 'PAS DE DEUX FROM FLAMES OF PARIS',
 'PAS DE DEUX FROM FLOWER FESTIVAL AT GENZANO',
 'PAS DE DEUX FROM GISELLE',
 'PAS DE DEUX FROM GRANDUATION BALL',
 'PAS DE DEUX FROM HARLEQUINADE',
 'PAS DE DEUX FROM HARLIQUENADE',
 'PAS DE DEUX FROM LA FILLE MAL GARDEE',
 'PAS DE DEUX FROM LE CORSAIRE',
 'PAS DE DEUX FROM NUTCRACKER',
 'PAS DE DEUX FROM PAQUITA',
 'PAS DE DEUX FROM SATANELLA',
 'PAS DE DEUX FROM TALISMAN',
 'PAS DE DEUX FROM 

In [47]:
dances = list(dancers[dancers["Category"].str.contains(r"ENSEMBLES|LARGE ENSEMBLES|SMALL ENSEMBLES|DUETS & TRIOS", na=False)].Name)

- Division / Position:
    - LARGE ENSEMBLES ; DANCES
    - SMALL ENSEMBLES ; DANCES AND NAMES
    - PAS DE DEUX ; DANCES AND NAMES
    - ENSEMBLES ; DANCES AND NAMES
    - DUET/TRIO ; DANCES

In [48]:
sorted(dancers[dancers["Division / Position"].str.contains(r"PAS DE DEUX", na=False)].Name.unique())

['ACE ACKET',
 'ADRIANA MARQUIQGUI CASADO',
 'AIDAN FUNG',
 'AINARA REINA PIQUERAS',
 'ALEJANDRO TIRADO ARADILLA',
 'ALESSANDRO MARINI',
 'ALEX GLADKOWSKI',
 'ALEX RODES TORA',
 'ALEXANDER VENS',
 'ALIANA NASSOS',
 'ALIANNA NASSOS',
 'ALICE LI',
 'AMARIS PONDER',
 'AMELIA WU',
 'ANATOLI BODROV',
 'ANDREW HALVERSON',
 'ANIYAH GUERRERO',
 'ANNA SURAPANENI',
 'ANTONELLA VILLA',
 'AQUA MARINE',
 'AURORA',
 'AVA KLOPPE',
 'AVA WHITEHURST',
 'BEN BIGELOW',
 'BEN HOLTON',
 'BEN MEYER',
 'BLAKELY BURNS',
 'BLUE BIRD SLEEPING BEAUTY',
 'BRAELYN SMITH',
 'BRAHMS VIOLIN CONCERTO',
 'BROOKLYN WHITLEY',
 'BRUNO COKELISS',
 'BRYCE YOUNG',
 'BRYNN TRAMLEY',
 'CAITLYN DIONEDA',
 'CAMILA ORTIZ MACIAS',
 'CANDICE YEO ZHI JING',
 'CAROLINE MCGOWAN',
 'CASIMIR FIRLIT',
 'CAYLON HULL',
 'CHAE YEON KIM',
 'CHARLEY TOSCANO',
 'CHARLOTTE BLAKE',
 'CHASING THE SAME MOMENT',
 "CHIARA D'ALO",
 'CHLOE FAN',
 'CLARA CRANE',
 'CLAUDIA TINARI',
 'CLAY KINLER',
 'CONSTANCE CABANES',
 'CONSTANCE MEEK',
 'DANIEL BERBER

In [49]:
dances += list(dancers[dancers["Division / Position"].str.contains(r"LARGE ENSEMBLES|DUET/TRIO", na=False)].Name)

- Dance Category:
    - ENSEMBLES ; MOSTLY DANCES AND NAMES
    - PAS DE DEUX ; DANCES AND NAMES
    - LARGE ENSEMBLES ; DANCES
    - SMALL ENSEMBLES ; DANCES
    - ENSEMBLES & PAS DE DEUX ; MOSTLY NAMES (KOREAN DANCERS) AND SOME DANCES
    - DUETS & PAS DE DEUX ; DANCES
    - DUET/TRIO ; DANCES

In [50]:
sorted(dancers[dancers["Dance Category"].str.contains("ENSEMBLES", na=False)].Name.unique())

['1LEDGE',
 '3 SISTERS',
 '3RD MOVEMENT FROM "PLAZA DEL FEUGO"',
 'A BOY IN LOVE',
 'A CELLO SONG DANCE',
 'A CHOPIN WALTZ',
 'A DANCE',
 'A DIFFERENT ANGLE',
 'A DIGA DOO',
 'A GLACIER',
 'A HUG TO CALL HOME',
 "A L'AREE DE KIRUNA",
 'A LIGHT IN THE DARKNESS',
 'A LITTLE CHOPIN MAZURKA',
 "A MIDSUMMER NIGHT'S DREAM - FAIRIES",
 "A QUEEN'S REIGN",
 'A SHORE NEVER REACHED',
 'A SONG FOR YOU',
 'A STORM INSIDE US, SILENCE OUTSIDE',
 'A STRANGE BEAUTY',
 'A STREAM WITHIN',
 'A STRONG FORCE',
 'A VAGUE SENSE OF PERSISTENCE',
 'A-FLAT',
 'ABSOLUTION',
 'ABSTRACT #17',
 'ACADEMY OF BALLET ARTS FL',
 "ACADEMY OF INT'L BALLET, PA",
 'ACADEMY OF RUSSIAN BALLET, VA',
 'ACT II PAS DE DEUX FROM GISELLE??',
 'ACTEON, PAS DE DEUX',
 'ADAGIO',
 'ADIR ADIRIM',
 'ADORATION',
 'ADRIFT',
 'AERIAL PORTRAIT',
 'AETHER',
 'AFRIKA',
 'AFTER THE FALL',
 'AFTER THE SHOCK',
 'AFTER THOUGHT',
 'AFTERNOON INTERLUDE',
 'AGGRANDIZMENT',
 'AIRBORNE',
 'AKHMEDOVA BALLET ACADEMY, MD',
 'AL SUON DI TARANTELLA',
 'ALEGR

In [57]:
sorted(dancers[dancers["Dance Category"].str.contains("PAS DE DEUX", na=False)].Name.unique())

['(11)',
 '(13)',
 '(14)',
 '(15)',
 '(16)',
 '(18)',
 '(9)',
 'ACTEON, PAS DE DEUX',
 'AFONSO FERREIRA (12), (LA FILLE MAL GARDEE)',
 'AFTERMATH',
 'ALBERTO GIL (13), (HARLEQUINADE)',
 'ALECSIA LAZARESCU',
 'ALEGRE',
 'ALESSANDRO FROLA, 12, MARTINA MANTOVANI, 13',
 'ALICE PELIZZA',
 'ALL PAS DE DEUX',
 'ALL THAT WAS LEFT',
 'ALLEGRO APPASSIONATO',
 'ANA COSTEIRA',
 'ANA FRANCISCA CRUZ',
 'ANA LUISA ARANTES NEGRAO',
 'ANA NOVAIS',
 'ANTÓNIO CASALINHO (16), (FLAMES OF PARIS',
 'AO WANG (17), LANG MA (19)',
 'ARE YOU GOING...?',
 'ARTHUR WILLIE (FLOWER FESTIVAL IN GENZANO)',
 'ASIER BAUTISTA (11), (SATANELLA)',
 'BACH PAS DE DEUX',
 'BEFORE I GO',
 'BEGIN THE BEGUINE',
 'BELLA KIRBY (11), AUSTEN ACEVEDO (13)',
 'BLUE BIRD PAS DE DEUX FROM SLEEPING BEAUTY',
 'BLUEBIRD PAS DE DEUX FROM SLEEPING BEAUTY',
 'BOYOUNG KIM 김보영',
 'BRIANNA BERRIOS (13), BLAKE KESSLER (14)',
 'BRUSH',
 'CAPULET',
 'CLAIRE RATHRUN (18), RAMMARU SHINDO (21)',
 'CLASSICAL BALLET OF WNY, NY',
 'COACTION AND CRANK HAND

In [52]:
d = dancers[dancers["Dance Category"].str.contains("ENSEMBLES & PAS DE DEUX", na=False)].Name.unique()
print(d)

dance_pattern = re.compile(r'^[^가-힣]+$') # keep anything that does NOT contain Korean characters
d = [x for x in d if dance_pattern.match(x)]

dances += d
d

['JIOH KIM 김지오' 'SIWOO JEONG 정시우' 'YUNJIN BAE 배윤진' 'JUMYEONG JANG 장주명'
 'JAEMYEONG JANG 장재명' 'SUNGYEON HONG 홍성연' 'SHIJEAN KIM 김시진'
 'SEOHYUN KIM 김서현' 'SEOYEON JEONG 정서연' 'GINA CHOI 최기나' 'NURI PARK 박누리'
 'SIHU BAN 반시후' 'HYEWON KANG 강혜원' 'SUNGMIN KIM 김성민' 'DOEUN KIM 김도은'
 'NAYEON KIM 김나연' 'BOYOUNG KIM 김보영' 'JIHYEONG CHOI 최지형' 'JIYOUNG KIM 김지영'
 'JIYU KWON 권지유' 'ALL ENSEMBLES' 'NOWHERE TOGETHER' 'WHISPER IN MY EAR']


['ALL ENSEMBLES', 'NOWHERE TOGETHER', 'WHISPER IN MY EAR']

In [53]:
dances += list(dancers[dancers["Dance Category"].str.contains(r"LARGE ENSEMBLES|SMALL ENSEMBLES|DUETS & PAS DE DEUX|DUET/TRIO", na=False)].Name)

In [54]:
dances = np.sort(np.unique(dances))
# dances

# Final Dancers data:

In [55]:
# name_instances = winners.Name_resolved.value_counts().to_dict()

In [56]:
dancers = dancers[
    (~dancers.Name.isin(dances))
]

dancers

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category,Names_resolved
6,"KORY GLATMAN, NJ",NaN,2000,"WASHINGTON, DC",3RD PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,[KORY GLATMAN]
7,ASHTON BERKLEY,"THE KIROV ACADEMY, DC",2000,"WASHINGTON, DC",2ND PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,NaN
8,DANA GAULE,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",1ST PLACE,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,NaN
9,KATRINA DALTON,"ALLEGRO DANCE ARTS ACADEMY, NJ, USA",2000,"WASHINGTON, DC",HOPE AWARD,PRE-COMPETITIVE AGE DIVISION,NaN,NaN,NaN,NaN,NaN
10,TIFFANY SMITH,"HARRISBUG DANCE CONSERVATORY, PA",2000,"WASHINGTON, DC",3RD PLACE (TIE),JUNIOR AGE DIVISION,CONTEMPORARY DANCE CATEGORY,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
43499,THE LAKE,"THE DANCE CENTER (TDC), CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"THE DANCE CENTER (TDC), CA",SMALL ENSEMBLES,NaN,NaN
43500,THE ORBITS,"EVA DANCE ACADEMY, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"EVA DANCE ACADEMY, CA",SMALL ENSEMBLES,NaN,NaN
43501,WITHOUT YOU,"JUN LU PERFORMING ARTS, CA",2026,"SAN FRANCISCO, CA (MAR)",TOP 12,NaN,NaN,"JUN LU PERFORMING ARTS, CA",SMALL ENSEMBLES,NaN,NaN
43518,KIRA FARGAS,FOR ALL THE WORK,2026,"SAN FRANCISCO, CA (MAR)",OUTSTANDING CHOREOGRAPHER AWARD,NaN,NaN,FOR ALL THE WORK,SPECIAL AWARDS,NaN,NaN
